# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR$^2$ dataset using the `mlcroissant` library, following the Croissant schema conventions.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nLicense: {metadata.license}")

## 2. Data Overview

Review available record sets, their fields, and corresponding entity `@id` values. All entities are referenced by their `@id` as per Croissant schema conventions.

In [ ]:
# List all record sets and their metadata
print("Available record sets and their @id values:")
record_sets = []
for rs in dataset.record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    record_sets.append(rs.id)

# For each record set, print its fields and their @id
for rs in dataset.record_sets:
    print(f"\nRecordSet: {rs.name} (@id={rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction

Load records from each record set into a DataFrame for analysis.

- Use the record set and field `@id`s as shown above.

In [ ]:
# Extract data from each record set into a DataFrame
# Use the list of record set @ids from the previous cell (variable: record_sets)
dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for record set @id={rs_id}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# For demonstration, choose the first loaded record set for statistics/EDA
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nAvailable columns for record set @id={first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No record set data loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering by numeric fields, normalization, and grouping data using relevant fields.

- Uses field `@id`s for all references.

In [ ]:
import numpy as np
# Choose a numeric field and a grouping field by their @id
# For illustration, list all possible numeric fields in the first record set
df = dataframes.get(first_rs_id)

if df is not None and not df.empty:
    print("Listing numeric fields (float/int):")
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().infer_objects().dtype, np.number)]
    print(numeric_fields)

    if numeric_fields:
        # Use the first numeric field for demonstration
        numeric_field_id = numeric_fields[0]

        threshold = df[numeric_field_id].mean() # e.g., use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # If the DataFrame contains categorical or groupable fields, group by the first such field (non-numeric)
        group_fields = [col for col in df.columns if col not in numeric_fields]
        if group_fields:
            group_field = group_fields[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}").reset_index()
                print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
                print(grouped_df.head())
    else:
        print("No numeric fields detected in the first record set.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize field distributions or relationships between fields in the dataset.
- Plots use columns referenced by their Croissant field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty and numeric_fields:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If grouped data is available, plot group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=f"mean_{numeric_field_id}", data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

- We loaded and explored the FAIR$^2$ dataset using `mlcroissant`.
- Record sets and field structures are provided via Croissant `@id` references for transparent provenance.
- Common EDA steps, such as filtering, normalization, and grouping, were illustrated on sample data.
- You can adapt this template for deeper analyses or visualizations using other fields or record sets.

For more information, review the Croissant metadata at the provided URL, and refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) for advanced usage.